# AI-Assisted Transition State Search

In this notebook, we compare classical and AI-based approaches for finding transition states (TS), with a focus on:

- Standard non-AI methods
- TS prediction (partially) using AI
- Ranking and refinement strategies with and without AI
- Practical workflows

We will:
1. Establish a classical baseline (NEB)
2. Exchange the electronic structure backend with an MLIP
3. Generate TS candidates using only AI (tsdiff)
4. Discuss refinement strategies and where AI can help

In [ ]:
import torch
import torchani
#import numpy as np

from ase import Atoms
from ase.mep import NEB, idpp_interpolate
#from ase.neb import NEB, idpp_interpolate
from ase.optimize import BFGS
from ase.calculators.calculator import Calculator, all_changes
#from ase.calculators.emt import EMT
from xtb.ase.calculator import XTB
#from ase.build import molecule

## Transition State Search: Practical Perspective

Classical methods:
- NEB: robust but expensive and initialization-sensitive

AI-based methods:
- Direct TS prediction: fast, but ranking is often imperfect

Key idea:
> The correct TS is often in the candidate set, but not ranked first.

We focus on:
- Candidate generation
- Ranking
- Efficient refinement

In [ ]:
# Structure definitions for ase for a proton transfer between two water molecules
reactant = Atoms(
    symbols=["O","H","H","O","H","H"],
    positions=[
        [0.000,0.000,0.000],
        [0.757,0.586,0.000],
        [-0.757,0.586,0.000],
        [2.500,0.000,0.000],
        [3.257,0.586,0.000],
        [1.743,0.586,0.000]
    ]
)

product = Atoms(
    symbols=["O","H","H","O","H","H"],
    positions=[
        [0.000,0.000,0.000],
        [0.757,0.586,0.000],
        [1.200,0.000,0.000],
        [2.500,0.000,0.000],
        [3.257,0.586,0.000],
        [1.743,0.586,0.000]
    ]
)

In [ ]:
# this is not a good example, because the xtb calculator doesn't converge,
# while the torchani calculator converges in 4 iterations
reactant2 = Atoms(
    symbols=["O","O","H","H","H","H","H"],
    positions=[
        [-1.200,  0.000,  0.000],  # O1 (hydronium)
        [ 1.200,  0.000,  0.000],  # O2 (water)

        [-1.600,  0.900,  0.000],  # H3
        [-1.600, -0.900,  0.000],  # H4

        [-0.300,  0.000,  0.000],  # H5 transferring proton

        [ 1.600,  0.900,  0.000],  # H6
        [ 1.600, -0.900,  0.000],  # H7
    ]
)

product2 = Atoms(
    symbols=["O","O","H","H","H","H","H"],
    positions=[
        [-1.200,  0.000,  0.000],  # O1 (water)
        [ 1.200,  0.000,  0.000],  # O2 (hydronium)

        [-1.600,  0.900,  0.000],  # H3
        [-1.600, -0.900,  0.000],  # H4

        [ 0.300,  0.000,  0.000],  # H5 transferred proton

        [ 1.600,  0.900,  0.000],  # H6
        [ 1.600, -0.900,  0.000],  # H7
    ]
)

In [ ]:
# raw form of standard organic SN2 reaction, which might be a good example later
"""
# SN2: Cl- + CH3Br → CH3Cl + Br-

from ase.build import molecule

ch3br = molecule("CH3Br")
cl = Atoms("Cl", positions=[[3.0, 0.0, 0.0]])

reactant = ch3br + cl

# Product: Cl replaces Br
product = reactant.copy()
product.positions[-1] = product.positions[0] + np.array([-1.8, 0, 0])

# Use xTB (already uses effective core approximations internally)

attach_calc(reactant)
attach_calc(product)

# Then same NEB workflow as above
"""

## Baseline: Nudged Elastic Band (NEB)

We compute a reaction path between reactant and product.

In [ ]:
# It took me a long time to find something, which can use neb with an electronic
# structure backend as well as an MLIP, and setting it up was even worse!

# Anyway this runs neb with xtb
def attach_xtb(atoms):
    #atoms.calc = EMT()
    atoms.calc = XTB(method="GFN2-xTB")
    return atoms

def run_neb(reactant, product, method, n_images=5):
    images = [reactant.copy()]

    # this didn't work, because this kind of simple interpolation
    # yielded guesses, which were so bad that neb didn't converge
    # even with small step sizes, etc.
    #for i in range(n_images):
    #    img = reactant.copy()
    #    img.positions = reactant.positions + (i+1)/(n_images+1)*(product.positions - reactant.positions)
    #    images.append(img)

    for i in range(n_images):
        images.append(reactant.copy())

    images.append(product.copy())

    # attach calculator ("attach" methods)
    for img in images:
        method(img)

    # For some reason different calculators are recommended, but
    # for the current setup this works as well
    neb = NEB(images, allow_shared_calculator=True)

    # This does some proper interpolation for the neb images
    idpp_interpolate(images)

    # choose a rather small step size here, because it won't converge otherwise
    opt = BFGS(neb, maxstep=0.02)

    # This will literally run forever, if you don't give a max_iter
    opt.run(fmax=0.1, steps=500)

    # assume middle image to be TS for now
    return images[len(images)//2], images

ts_xtb, neb_images_xtb = run_neb(reactant, product, attach_xtb)
# TODO: output and visualize structure

In [ ]:
# And the following two cells run neb with an MLIP

# setup MLIP model
# if your device has a GPU I highly recommend to use it, by setting the device here
# and previously importing the appropriate pytorch expansion into your conda environment.
# For an nvidia GPU this would be torch.device("cuda")
device = torch.device("cpu")
ani_model = torchani.models.ANI2x().to(device)
species_converter = ani_model.species_converter

# mocked calculator class for MLIP
class ANICalculator(Calculator):
    implemented_properties = ['energy', 'forces']

    def calculate(self, atoms=None, properties=['energy'], system_changes=all_changes):
        super().calculate(atoms, properties, system_changes)

        coords = torch.tensor(
            atoms.get_positions(),
            dtype=torch.float32,
            device=device,
            requires_grad=True
        ).unsqueeze(0)

        atomic_numbers = torch.tensor(
            [atoms.get_atomic_numbers()],
            dtype=torch.long,
            device=device
        )

        energy = ani_model((atomic_numbers, coords)).energies
        forces = -torch.autograd.grad(energy.sum(), coords)[0]

        self.results['energy'] = energy.item()
        self.results['forces'] = forces.squeeze(0).cpu().detach().numpy()


def attach_ani(atoms):
    atoms.calc = ANICalculator()
    return atoms

In [ ]:
# run neb with MLIP
ts_ani, neb_images_ani = run_neb(reactant, product, attach_ani)
# TODO: output and visualize structure

## Second example

In [ ]:
ts_xtb, neb_images_xtb = run_neb(reactant2, product2, attach_xtb)

In [ ]:
ts_ani, neb_images_ani = run_neb(reactant2, product2, attach_ani)

As can be seen the performances are very different, stressing the importance of the electronic structure backend and especially the initial guess. This is a big downside, so models providing a proper guess structure are very important even for seemingly simple cases!

### Refinement of the guessed structures

Standard: Use TS guess and optimize purely with electronic structure backend

In [ ]:
def refine_xtb(ts):
    ts = ts.copy()
    attach_xtb(ts)
    opt = BFGS(ts)
    opt.run(fmax=0.02)
    return ts

#ts_refined_std = refine_xtb(ranked[0][0])

Surrogate accelaration by preoptimizing with MLIP

In [ ]:
def refine_ml_then_xtb(ts):
    ts = ts.copy()
    
    # Step 1: ANI refinement
    attach_ani(ts)
    opt = BFGS(ts)
    opt.run(fmax=0.02)
    
    # Step 2: xTB refinement
    attach_xtb(ts)
    opt = BFGS(ts)
    opt.run(fmax=0.02)
    
    return ts

#ts_refined_ml = refine_ml_then_xtb(ranked[0][0])

Accelaration of initial Hessian guess (using MLIP)

In [ ]:
# For this I found that Hessians are typically build from MLIP information

## Ranking TS Candidates

We rank candidates based on approximate energies or heuristics.

Observation:
- Top-1 is sometimes wrong
- Correct TS may appear in top-2 or top-3

Hence, when you predict a transition state, you should always generate multiple candidates.